# 03a — Base Features

Builds the was_offered grid and all features that do not require past data.

**Input:**  `data/processed/sfu_clean.db`, `data/raw/sfu_ml.db`
**Output:** `data/processed/03a_train.csv`, `data/processed/03a_test.csv`

**Output columns:**
`ml_course_id, ml_term_id, was_offered, year, term, term_order, semester_code,
is_covid_affected, course_level, is_grad, prereq_count, has_coreqs, units,
has_designation, is_online, is_evening, is_burnaby, is_surrey, is_harbour_ctr,
is_other_van, is_off_campus`

## 0. Setup

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product

CLEAN_DB  = Path('../data/processed/sfu_clean.db')
RAW_DB    = Path('../data/raw/sfu_ml.db')
OUT_PATH  = Path('../data/processed')

assert CLEAN_DB.exists(), f'not found: {CLEAN_DB}'
assert RAW_DB.exists(),   f'not found: {RAW_DB}'
OUT_PATH.mkdir(exist_ok=True)
print('ready')

ready


## 1. Load offerings

In [2]:
conn = sqlite3.connect(CLEAN_DB)
df   = pd.read_sql('SELECT * FROM offerings', conn)
conn.close()

print(f'{len(df):,} rows  x  {df.shape[1]} cols')
print(f'columns: {list(df.columns)}')

33,168 rows  x  18 cols
columns: ['offering_id', 'ml_course_id', 'ml_term_id', 'dept_code', 'course_number', 'section_code', 'instructor', 'campus', 'capacity', 'enrolled', 'course_level', 'degree_level', 'title', 'year', 'term', 'term_order', 'semester_code', 'is_covid_affected']


## 2. Define split and create train_raw / test_raw

In [3]:
TRAIN_YEAR = 2024
TRAIN_TERM = 3       # term_order 3 = fall

train_mask = (
    (df['year'] < TRAIN_YEAR) |
    ((df['year'] == TRAIN_YEAR) & (df['term_order'] <= TRAIN_TERM))
)

train_raw = df[train_mask].copy()
test_raw  = df[~train_mask].copy()

print(f'train_raw: {len(train_raw):,} rows')
print(f'test_raw:  {len(test_raw):,} rows')
print()
print('train terms:')
print(train_raw.groupby(['year','term']).size().reset_index(name='n').sort_values(['year','term']).to_string(index=False))
print()
print('test terms:')
print(test_raw.groupby(['year','term']).size().reset_index(name='n').sort_values(['year','term']).to_string(index=False))

train_raw: 27,327 rows
test_raw:  5,841 rows

train terms:
 year   term    n
 2020   fall 1966
 2020 spring 2009
 2020 summer 1324
 2021   fall 2044
 2021 spring 2070
 2021 summer 1322
 2022   fall 2005
 2022 spring 2069
 2022 summer 1380
 2023   fall 2123
 2023 spring 2102
 2023 summer 1336
 2024   fall 2077
 2024 spring 2116
 2024 summer 1384

test terms:
 year   term    n
 2025   fall 2206
 2025 spring 2235
 2025 summer 1400


## 3. Build was_offered grid

Every course × every term → was_offered 0/1.
A course that ran in a term has was_offered=1. Absence = 0.

In [4]:
# unique courses and terms from the full dataset
all_courses  = df['ml_course_id'].dropna().unique()
all_term_ids = df['ml_term_id'].unique()

print(f'unique courses:  {len(all_courses):,}')
print(f'unique terms:    {len(all_term_ids)}')
print(f'grid size:       {len(all_courses) * len(all_term_ids):,}')

unique courses:  3,184
unique terms:    18
grid size:       57,312


In [5]:
# build full grid
grid = pd.DataFrame(
    list(product(all_courses, all_term_ids)),
    columns=['ml_course_id', 'ml_term_id']
)

# which combinations had at least one section
offered_set = set(zip(df['ml_course_id'].dropna(), df['ml_term_id']))
grid['was_offered'] = [
    1 if (cid, tid) in offered_set else 0
    for cid, tid in zip(grid['ml_course_id'], grid['ml_term_id'])
]

# join term metadata from df
term_meta = (
    df[['ml_term_id','year','term','term_order','semester_code','is_covid_affected']]
    .drop_duplicates()
)
grid = grid.merge(term_meta, on='ml_term_id', how='left')

print(f'grid: {len(grid):,} rows')
print(f'was_offered=1: {grid["was_offered"].sum():,}  ({grid["was_offered"].mean()*100:.1f}%)')
print(f'was_offered=0: {(grid["was_offered"]==0).sum():,}')

grid: 57,312 rows
was_offered=1: 24,083  (42.0%)
was_offered=0: 33,229


In [6]:
# split grid using same boundary as train_raw / test_raw
grid_mask = (
    (grid['year'] < TRAIN_YEAR) |
    ((grid['year'] == TRAIN_YEAR) & (grid['term_order'] <= TRAIN_TERM))
)
grid_train = grid[grid_mask].copy()
grid_test  = grid[~grid_mask].copy()

print(f'grid_train: {len(grid_train):,} rows  was_offered rate: {grid_train["was_offered"].mean()*100:.1f}%')
print(f'grid_test:  {len(grid_test):,} rows   was_offered rate: {grid_test["was_offered"].mean()*100:.1f}%')

grid_train: 47,760 rows  was_offered rate: 41.4%
grid_test:  9,552 rows   was_offered rate: 45.1%


## 4. Static course features

`course_level` and `degree_level` from sfu_clean.db.
`prereq_count`, `has_coreqs`, `units`, `designation` from sfu_ml.db (not in clean schema).

In [7]:
# from clean DB — one row per course
course_base = (
    df[['ml_course_id','course_level','degree_level']]
    .drop_duplicates(subset='ml_course_id')
    .copy()
)

# from raw DB — extra course metadata
conn_raw   = sqlite3.connect(RAW_DB)
course_extra = pd.read_sql("""
    SELECT ml_course_id, prereq_count, has_coreqs, units, designation
    FROM ml_courses
""", conn_raw)
conn_raw.close()

# merge
courses_meta = course_base.merge(course_extra, on='ml_course_id', how='left')

# derive flags
courses_meta['is_grad']          = (courses_meta['degree_level'] == 'GRAD').astype(int)
courses_meta['has_designation']  = courses_meta['designation'].notna().astype(int)

# fill nulls with sensible defaults
courses_meta['prereq_count'] = courses_meta['prereq_count'].fillna(0).astype(int)
courses_meta['has_coreqs']   = courses_meta['has_coreqs'].fillna(0).astype(int)
courses_meta['units']        = courses_meta['units'].fillna(3).astype(int)

# drop raw columns we derived from
courses_meta = courses_meta.drop(columns=['degree_level','designation'])

print(f'courses_meta: {len(courses_meta):,} courses  x  {courses_meta.shape[1]} cols')
print(f'columns: {list(courses_meta.columns)}')
courses_meta.head(3)

courses_meta: 3,184 courses  x  7 cols
columns: ['ml_course_id', 'course_level', 'prereq_count', 'has_coreqs', 'units', 'is_grad', 'has_designation']


,ml_course_id,course_level,prereq_count,has_coreqs,units,is_grad,has_designation
0,3,300,1,0,3,0,0
1,4,300,1,0,3,0,0
2,5,300,2,0,3,0,1


## 5. Delivery features

One row per course × term. Uses the mode of section_code prefix and campus across all sections for that course-term.

In [8]:
def mode_val(s):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else None

delivery = (
    df
    .groupby(['ml_course_id','ml_term_id'])
    .agg(
        section_prefix = ('section_code', lambda s: s.str[0].mode().iloc[0]),
        campus         = ('campus',        mode_val)
    )
    .reset_index()
)

# section code flags
delivery['is_online']  = (delivery['section_prefix'] == 'C').astype(int)
delivery['is_evening'] = (delivery['section_prefix'] == 'E').astype(int)

# campus flags — Burnaby is implicit when all 0
delivery['is_burnaby']     = (delivery['campus'] == 'Burnaby').astype(int)
delivery['is_surrey']      = (delivery['campus'] == 'Surrey').astype(int)
delivery['is_harbour_ctr'] = (delivery['campus'] == 'Harbour Ctr').astype(int)
delivery['is_other_van']   = (delivery['campus'] == 'Other Vancouver').astype(int)
delivery['is_off_campus']  = (delivery['campus'].isin(['Off-campus','Great North. Way'])).astype(int)

delivery = delivery.drop(columns=['section_prefix','campus'])

print(f'delivery: {len(delivery):,} course-term rows  x  {delivery.shape[1]} cols')
delivery.head(3)

delivery: 24,083 course-term rows  x  9 cols


,ml_course_id,ml_term_id,is_online,is_evening,is_burnaby,is_surrey,is_harbour_ctr,is_other_van,is_off_campus
0,1,3,0,1,1,0,0,0,0
1,1,6,0,1,1,0,0,0,0
2,1,9,0,1,1,0,0,0,0


## 6. Assemble and save

In [9]:
def assemble_base(grid_part, courses_meta_df, delivery_df):
    out = grid_part.copy()

    # join static course features
    out = out.merge(courses_meta_df, on='ml_course_id', how='left')

    # join delivery features
    out = out.merge(delivery_df, on=['ml_course_id','ml_term_id'], how='left')

    # fill delivery flags with 0 for course-term pairs not in clean DB
    flag_cols = [
        'is_online','is_evening',
        'is_burnaby','is_surrey','is_harbour_ctr','is_other_van','is_off_campus'
    ]
    out[flag_cols] = out[flag_cols].fillna(0).astype(int)

    return out

train_base = assemble_base(grid_train, courses_meta, delivery)
test_base  = assemble_base(grid_test,  courses_meta, delivery)

print(f'train_base: {train_base.shape}')
print(f'test_base:  {test_base.shape}')

train_base: (47760, 21)
test_base:  (9552, 21)


In [10]:
# verify expected columns
expected_cols = [
    'ml_course_id','ml_term_id','was_offered',
    'year','term','term_order','semester_code','is_covid_affected',
    'course_level','is_grad','prereq_count','has_coreqs','units','has_designation',
    'is_online','is_evening',
    'is_burnaby','is_surrey','is_harbour_ctr','is_other_van','is_off_campus'
]
missing = [c for c in expected_cols if c not in train_base.columns]
extra   = [c for c in train_base.columns if c not in expected_cols]
print(f'missing cols: {missing if missing else "none"}')
print(f'extra cols:   {extra if extra else "none"}')
print()
# nulls
nulls = train_base.isnull().sum()
print('null counts:')
print(nulls[nulls > 0] if nulls.any() else 'none')

missing cols: none
extra cols:   none

null counts:
none


In [11]:
# spot check: CMPT 225
cmpt225_id = df[(df['dept_code']=='CMPT') & (df['course_number']=='225')]['ml_course_id'].unique()
check = train_base[train_base['ml_course_id'].isin(cmpt225_id)]
print(f'CMPT 225 rows in train grid: {len(check)}')
check[['year','term','was_offered','course_level','is_grad','is_online','is_burnaby']].sort_values(['year','term'])

CMPT 225 rows in train grid: 15


,year,term,was_offered,course_level,is_grad,is_online,is_burnaby
5912,2020,fall,1,200,0,0,1
5910,2020,spring,1,200,0,0,1
5911,2020,summer,1,200,0,0,1
5915,2021,fall,1,200,0,0,1
5913,2021,spring,1,200,0,0,1
5914,2021,summer,1,200,0,0,1
5918,2022,fall,1,200,0,0,1
5916,2022,spring,1,200,0,0,1
5917,2022,summer,1,200,0,0,1
5921,2023,fall,1,200,0,0,1


In [12]:
# save
train_base.to_csv(OUT_PATH / '03a_train.csv', index=False)
test_base.to_csv( OUT_PATH / '03a_test.csv',  index=False)

print(f'saved 03a_train.csv: {len(train_base):,} rows  x  {train_base.shape[1]} cols')
print(f'saved 03a_test.csv:  {len(test_base):,} rows   x  {test_base.shape[1]} cols')

saved 03a_train.csv: 47,760 rows  x  21 cols
saved 03a_test.csv:  9,552 rows   x  21 cols


In [13]:
# read back to confirm
tr = pd.read_csv(OUT_PATH / '03a_train.csv')
te = pd.read_csv(OUT_PATH / '03a_test.csv')
assert tr.shape == train_base.shape, f'shape mismatch: {tr.shape} vs {train_base.shape}'
assert te.shape == test_base.shape,  f'shape mismatch: {te.shape} vs {test_base.shape}'
print('verified')
print(f'columns: {list(tr.columns)}')

verified
columns: ['ml_course_id', 'ml_term_id', 'was_offered', 'year', 'term', 'term_order', 'semester_code', 'is_covid_affected', 'course_level', 'prereq_count', 'has_coreqs', 'units', 'is_grad', 'has_designation', 'is_online', 'is_evening', 'is_burnaby', 'is_surrey', 'is_harbour_ctr', 'is_other_van', 'is_off_campus']
